# Laser Microphone - Train on Kaggle (free GPU)

This notebook trains the digit LSTM on Kaggle's **free GPU**, which turns the ~20-minute laptop CPU run into ~1-2 minutes. It reuses the **exact same** `src/` code - no logic is copied here, so results match the local pipeline.

## One-time setup (do this before running)
1. **Upload the code as a Kaggle Dataset**
   - On kaggle.com: *Datasets -> New Dataset* -> upload your `laser_microphone_ml` folder (or at least the `src/` folder). Give it a name, e.g. `laser-microphone-ml`.
2. **Create/open a Notebook** and on the right panel:
   - *Add Input* -> add the dataset you just created.
   - *Settings -> Accelerator* -> **GPU T4 x2** (or P100).
   - *Settings -> Internet* -> **On** (needed to download the FSDD audio).
3. Run all cells top to bottom.

Outputs (trained model + confusion matrix) are written to `/kaggle/working/` and appear under the notebook's **Output** tab for download.

> Prefer not to enable Internet? Instead add the public **"Free Spoken Digit Dataset"** Kaggle dataset as a second input, and set `RAW_DIR` in the data cell to its `recordings` folder.

In [ ]:
# 1) Confirm we actually have a GPU.
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU - go to Settings -> Accelerator -> GPU, then restart the kernel.')

In [ ]:
# 2) torchaudio ships on Kaggle; we just need soundfile for reading WAVs.
!pip install -q soundfile

In [ ]:
# 3) Point the pipeline at Kaggle folders, and find our uploaded src/ code.
#    IMPORTANT: set these environment variables BEFORE importing config/train,
#    because config.py reads them at import time.
import os, sys
from pathlib import Path

RAW_DIR = '/kaggle/working/data/raw'          # where we will put the WAV files
os.environ['LMML_RAW_DIR'] = RAW_DIR          # input WAVs
os.environ['LMML_OUTPUT_DIR'] = '/kaggle/working'  # models/ + results/ (writable)

# Auto-locate the src/ folder inside whatever dataset you attached.
src_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.py' in files and 'config.py' in files:
        src_dir = root
        break
assert src_dir, 'Could not find src/ (train.py, config.py) under /kaggle/input. Add your code dataset as an Input.'
sys.path.append(src_dir)
print('Using pipeline code from:', src_dir)

In [ ]:
# 4) Download the Free Spoken Digit Dataset (3000 WAVs) into RAW_DIR.
#    Requires Internet = On. Skip this cell if you attached an FSDD dataset
#    instead and pointed LMML_RAW_DIR at its recordings folder.
import shutil, subprocess
os.makedirs(RAW_DIR, exist_ok=True)
if not any(Path(RAW_DIR).glob('*.wav')):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Jakobovski/free-spoken-digit-dataset.git',
                    '/kaggle/working/fsdd'], check=True)
    for wav in Path('/kaggle/working/fsdd/recordings').glob('*.wav'):
        shutil.copy2(wav, Path(RAW_DIR) / wav.name)
print('WAV files ready:', len(list(Path(RAW_DIR).glob('*.wav'))))

In [ ]:
# 5) Train. This imports the SAME train.py used locally and runs it on the GPU.
#    (Optional: speed it up further by lowering epochs, e.g. `config.NUM_EPOCHS = 25`.)
import config, train
print('Training device:', config.DEVICE)   # should say cuda
train.main()

In [ ]:
# 6) Evaluate on the held-out test set (accuracy, per-class, confusion matrix).
import evaluate
evaluate.main()

In [ ]:
# 7) Show the confusion matrix and confirm the saved files.
from IPython.display import Image, display
cm = '/kaggle/working/results/plots/confusion_matrix.png'
if os.path.exists(cm):
    display(Image(cm))
print('Trained model:', '/kaggle/working/models/best_model.pt')
print('Download it from the notebook Output tab, then place it in your local models/ folder.')

## After training
- Download `models/best_model.pt` from the **Output** tab and drop it into your local `models/` folder.
- Then run the mic UI locally: `python src/app.py`.

**Laser note:** when real laser data exists, upload it as a Kaggle dataset (WAV or CSV converted with `scripts/wav_from_csv.py`), point `LMML_RAW_DIR` at it, and re-run - the GPU training path is identical.